In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # 🏛️ 04 · Ownership & Stewardship — Unity Catalog
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Aplicar y validar metadata de **ownership, stewardship y custodia técnica**
# MAGIC en los activos críticos del ciclo de datos YelpConnect.
# MAGIC
# MAGIC La asignación se deriva del modelo de gobierno/RACI y se expresa mediante
# MAGIC tags estables de Unity Catalog:
# MAGIC
# MAGIC - `business_owner`
# MAGIC - `data_steward`
# MAGIC - `technical_owner`
# MAGIC - `data_domain`
# MAGIC - `data_layer`
# MAGIC - `data_product`
# MAGIC
# MAGIC Además se registran:
# MAGIC
# MAGIC - `governance_model = raci_v1_4`
# MAGIC - `ownership_source = raci_capabilities`
# MAGIC
# MAGIC ## Principio de diseño
# MAGIC
# MAGIC **La RACI es la fuente de verdad del ownership.**
# MAGIC
# MAGIC `business_owner`, `data_steward` y `technical_owner` son metadata de
# MAGIC gobierno. No deben confundirse con el `OWNER TO` nativo de Unity Catalog,
# MAGIC que representa propiedad técnica del objeto.
# MAGIC
# MAGIC ## Alcance
# MAGIC
# MAGIC - Bronze
# MAGIC - Silver
# MAGIC - Gobierno (`yelp_gov`)
# MAGIC - Gold
# MAGIC
# MAGIC Este notebook **no transforma datos**, **no recalcula riesgo** y
# MAGIC **no modifica la lógica T&S**.

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"
SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"
GOLD_SCHEMA = "yelp_gold"

BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

PIPELINE_NAME = "04_ownership_stewardship_uc"
GOVERNANCE_MODEL = "raci_v1_4"
OWNERSHIP_SOURCE = "raci_capabilities"

REQUIRED_GOVERNANCE_TAGS = [
    "data_domain",
    "data_layer",
    "data_product",
    "business_owner",
    "data_steward",
    "technical_owner"
]

ADDITIONAL_GOVERNANCE_TAGS = [
    "governance_model",
    "ownership_source"
]

print("=" * 88)
print("YELPCONNECT - OWNERSHIP & STEWARDSHIP UNITY CATALOG")
print("=" * 88)
print(f"Pipeline          : {PIPELINE_NAME}")
print(f"Catalog           : {CATALOG}")
print(f"Governance model  : {GOVERNANCE_MODEL}")
print(f"Ownership source  : {OWNERSHIP_SOURCE}")
print("=" * 88)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Convención de roles
# MAGIC
# MAGIC Los valores almacenados en tags son **roles funcionales estables**, no nombres de personas.
# MAGIC
# MAGIC | Rol | Significado |
# MAGIC |---|---|
# MAGIC | `business_owner` | Accountable funcional del dato o producto |
# MAGIC | `data_steward` | Responsable operativo de metadata, calidad y definición |
# MAGIC | `technical_owner` | Responsable técnico/custodio del activo |

# COMMAND ----------

# ============================================================
# 3. MATRIZ DE OWNERSHIP / STEWARDSHIP
# ============================================================

GOVERNANCE_MATRIX = {

    # BRONZE
    f"{BRONZE}.brz_business": {
        "data_domain": "business",
        "data_layer": "bronze",
        "data_product": "business_raw",
        "business_owner": "product_business",
        "data_steward": "business_steward",
        "technical_owner": "data_engineering"
    },

    f"{BRONZE}.brz_user": {
        "data_domain": "identity",
        "data_layer": "bronze",
        "data_product": "user_raw",
        "business_owner": "product_identity",
        "data_steward": "identity_steward",
        "technical_owner": "data_engineering"
    },

    f"{BRONZE}.brz_review": {
        "data_domain": "content",
        "data_layer": "bronze",
        "data_product": "review_raw",
        "business_owner": "product_content",
        "data_steward": "content_steward",
        "technical_owner": "data_engineering"
    },

    f"{BRONZE}.brz_tip": {
        "data_domain": "interaction",
        "data_layer": "bronze",
        "data_product": "tip_raw",
        "business_owner": "product_interaction",
        "data_steward": "interaction_steward",
        "technical_owner": "data_engineering"
    },

    f"{BRONZE}.brz_checkin": {
        "data_domain": "interaction",
        "data_layer": "bronze",
        "data_product": "checkin_raw",
        "business_owner": "product_interaction",
        "data_steward": "interaction_steward",
        "technical_owner": "data_engineering"
    },

    # SILVER BASE
    f"{SILVER}.slv_business": {
        "data_domain": "business",
        "data_layer": "silver",
        "data_product": "business",
        "business_owner": "product_business",
        "data_steward": "business_steward",
        "technical_owner": "data_engineering"
    },

    f"{SILVER}.slv_user": {
        "data_domain": "identity",
        "data_layer": "silver",
        "data_product": "user",
        "business_owner": "product_identity",
        "data_steward": "identity_steward",
        "technical_owner": "data_engineering"
    },

    f"{SILVER}.slv_review": {
        "data_domain": "content",
        "data_layer": "silver",
        "data_product": "review",
        "business_owner": "product_content",
        "data_steward": "content_steward",
        "technical_owner": "data_engineering"
    },

    # SILVER T&S
    f"{SILVER}.slv_review_features": {
        "data_domain": "trust_safety",
        "data_layer": "silver",
        "data_product": "review_features",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{SILVER}.slv_review_evaluation": {
        "data_domain": "trust_safety",
        "data_layer": "silver",
        "data_product": "review_evaluation",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{SILVER}.slv_review_risk": {
        "data_domain": "trust_safety",
        "data_layer": "silver",
        "data_product": "review_risk",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    # GOVERNANCE / DQ
    f"{GOV}.dq_reglas": {
        "data_domain": "data_governance",
        "data_layer": "governance",
        "data_product": "dq_rule_catalog",
        "business_owner": "data_governance",
        "data_steward": "data_quality_steward",
        "technical_owner": "data_platform"
    },

    f"{GOV}.dq_resultados": {
        "data_domain": "data_governance",
        "data_layer": "governance",
        "data_product": "dq_results",
        "business_owner": "data_governance",
        "data_steward": "data_quality_steward",
        "technical_owner": "data_engineering"
    },

    # GOVERNANCE / T&S
    f"{GOV}.evaluation_rule": {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_rule_catalog",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_platform"
    },

    f"{GOV}.evaluation_rule_version": {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_rule_version",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_platform"
    },

    f"{GOV}.evaluation_run": {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_run",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_platform"
    },

    # GOLD
    f"{GOLD}.gld_dim_business": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "dim_business",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_dim_user": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "dim_user",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_dim_rule": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "dim_rule",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_dim_evaluation_run": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "dim_evaluation_run",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_fact_review_evaluation": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "review_evaluation_fact",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_review_risk": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "review_risk",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_investigation_queue": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "investigation_queue",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_kpi_trust_safety_overview": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "kpi_trust_safety_overview",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_kpi_trust_safety_business": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "kpi_trust_safety_business",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_kpi_trust_safety_daily": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "kpi_trust_safety_daily",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    },

    f"{GOLD}.gld_kpi_trust_safety_rule": {
        "data_domain": "trust_safety",
        "data_layer": "gold",
        "data_product": "kpi_trust_safety_rule",
        "business_owner": "trust_safety",
        "data_steward": "trust_safety_steward",
        "technical_owner": "data_engineering"
    }
}

for table_name, tags in GOVERNANCE_MATRIX.items():
    tags["governance_model"] = GOVERNANCE_MODEL
    tags["ownership_source"] = OWNERSHIP_SOURCE

print(f"Activos gobernados configurados: {len(GOVERNANCE_MATRIX)}")

# COMMAND ----------

# ============================================================
# 4. VALIDAR ACTIVOS REQUERIDOS
# ============================================================

missing_tables = [
    table_name
    for table_name in GOVERNANCE_MATRIX
    if not spark.catalog.tableExists(table_name)
]

if missing_tables:

    print("❌ Activos requeridos no encontrados:")

    for table_name in missing_tables:
        print(f" - {table_name}")

    raise Exception(
        "No se puede aplicar ownership/stewardship: "
        "faltan activos requeridos."
    )

print(f"✔ {len(GOVERNANCE_MATRIX)} activos requeridos disponibles.")

# COMMAND ----------

# ============================================================
# 5. VALIDAR CONTRATO DE TAGS
# ============================================================

invalid_contract = []

for table_name, tags in GOVERNANCE_MATRIX.items():

    for required_tag in REQUIRED_GOVERNANCE_TAGS:

        value = tags.get(required_tag)

        if value is None or str(value).strip() == "":
            invalid_contract.append((table_name, required_tag))

if invalid_contract:

    print("❌ Contrato de gobierno incompleto:")

    for table_name, tag_name in invalid_contract:
        print(f" - {table_name}: falta {tag_name}")

    raise Exception(
        "La matriz de ownership no cumple "
        "el contrato mínimo de gobierno."
    )

print("✔ Contrato mínimo de ownership/stewardship completo.")

# COMMAND ----------

# ============================================================
# 6. HELPERS
# ============================================================

def escape_tag_value(value):
    return str(value).replace("'", "''")


def apply_table_tags(table_name, tags):

    tags_sql = ",\n                ".join(
        f"'{escape_tag_value(key)}' = "
        f"'{escape_tag_value(value)}'"
        for key, value in tags.items()
    )

    spark.sql(
        f"""
        ALTER TABLE {table_name}
        SET TAGS (
                {tags_sql}
        )
        """
    )

# COMMAND ----------

# ============================================================
# 7. APLICAR OWNERSHIP / STEWARDSHIP
# ============================================================

applied_rows = []

for table_name, tags in GOVERNANCE_MATRIX.items():

    apply_table_tags(table_name, tags)

    for tag_name, tag_value in tags.items():
        applied_rows.append(
            (table_name, tag_name, tag_value)
        )

    print(f"✔ Governance tags aplicados: {table_name}")

print()
print("✔ Ownership/Stewardship aplicado a todos los activos configurados.")

# COMMAND ----------

# ============================================================
# 8. EVIDENCIA ESPERADA
# ============================================================

applied_df = spark.createDataFrame(
    applied_rows,
    [
        "table_name",
        "tag_name",
        "expected_value"
    ]
)

display(
    applied_df.orderBy(
        "table_name",
        "tag_name"
    )
)

# COMMAND ----------

# ============================================================
# 9. TAGS OBSERVADOS EN UNITY CATALOG
# ============================================================

table_tags = spark.sql(
    f"""
    SELECT
        schema_name,
        table_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.table_tags
    WHERE schema_name IN (
        '{BRONZE_SCHEMA}',
        '{SILVER_SCHEMA}',
        '{GOV_SCHEMA}',
        '{GOLD_SCHEMA}'
    )
      AND tag_name IN (
        'data_domain',
        'data_layer',
        'data_product',
        'business_owner',
        'data_steward',
        'technical_owner',
        'governance_model',
        'ownership_source'
      )
    """
)

display(
    table_tags.orderBy(
        "schema_name",
        "table_name",
        "tag_name"
    )
)

# COMMAND ----------

# ============================================================
# 10. MATRIZ ESPERADA VS OBSERVADA
# ============================================================

expected = {}

for full_table_name, tags in GOVERNANCE_MATRIX.items():

    parts = full_table_name.split(".")
    schema_name = parts[1]
    short_table_name = parts[2]

    for tag_name, tag_value in tags.items():
        expected[
            (
                schema_name,
                short_table_name,
                tag_name
            )
        ] = tag_value


observed = {
    (
        row.schema_name,
        row.table_name,
        row.tag_name
    ): row.tag_value
    for row in table_tags.collect()
}


validation_rows = []

for key, expected_value in expected.items():

    actual_value = observed.get(key)
    passed = (actual_value == expected_value)

    validation_rows.append(
        (
            key[0],
            key[1],
            key[2],
            expected_value,
            actual_value,
            passed
        )
    )


validation_df = spark.createDataFrame(
    validation_rows,
    [
        "schema_name",
        "table_name",
        "tag_name",
        "expected_value",
        "actual_value",
        "passed"
    ]
)

display(
    validation_df.orderBy(
        "schema_name",
        "table_name",
        "tag_name"
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Quality Gate de Ownership & Stewardship
# MAGIC
# MAGIC Un activo no se considera gobernado únicamente porque exista.
# MAGIC Para superar este gate, cada activo configurado debe contener exactamente
# MAGIC los tags definidos por la RACI.

# COMMAND ----------

# ============================================================
# 11. QUALITY GATE AUTOSUFICIENTE
# ============================================================

table_tags_gate = spark.sql(
    f"""
    SELECT
        schema_name,
        table_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.table_tags
    WHERE schema_name IN (
        '{BRONZE_SCHEMA}',
        '{SILVER_SCHEMA}',
        '{GOV_SCHEMA}',
        '{GOLD_SCHEMA}'
    )
      AND tag_name IN (
        'data_domain',
        'data_layer',
        'data_product',
        'business_owner',
        'data_steward',
        'technical_owner',
        'governance_model',
        'ownership_source'
      )
    """
)

observed_gate = {
    (
        row.schema_name,
        row.table_name,
        row.tag_name
    ): row.tag_value
    for row in table_tags_gate.collect()
}

validation_rows_gate = []

for key, expected_value in expected.items():

    actual_value = observed_gate.get(key)
    passed = (actual_value == expected_value)

    validation_rows_gate.append(
        (
            key[0],
            key[1],
            key[2],
            expected_value,
            actual_value,
            passed
        )
    )

validation_gate_df = spark.createDataFrame(
    validation_rows_gate,
    [
        "schema_name",
        "table_name",
        "tag_name",
        "expected_value",
        "actual_value",
        "passed"
    ]
)

failed_tags = (
    validation_gate_df
    .filter(
        F.col("passed") == False
    )
    .count()
)

total_tags = validation_gate_df.count()
passed_tags = total_tags - failed_tags
governed_assets = len(GOVERNANCE_MATRIX)

print()
print("=" * 88)
print("RESULTADO OWNERSHIP & STEWARDSHIP UNITY CATALOG")
print("=" * 88)
print(f"Activos gobernados         : {governed_assets}")
print(f"Tags esperados             : {total_tags}")
print(f"Tags correctos             : {passed_tags}")
print(f"Tags incorrectos/faltantes : {failed_tags}")
print("=" * 88)

if failed_tags > 0:

    print()
    print("❌ Tags de ownership/stewardship incorrectos o faltantes:")

    display(
        validation_gate_df
        .filter(
            F.col("passed") == False
        )
        .orderBy(
            "schema_name",
            "table_name",
            "tag_name"
        )
    )

    raise Exception(
        f"""
        ❌ QUALITY GATE DE OWNERSHIP / STEWARDSHIP FALLÓ.

        Tags incorrectos o faltantes:
        {failed_tags}
        """
    )

print()
print("✔ QUALITY GATE DE OWNERSHIP / STEWARDSHIP SUPERADO")
print("✔ La metadata efectiva de Unity Catalog coincide con la RACI.")
print("✔ Siguiente etapa: 07_validacion_end_to_end")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. OWNER nativo de Unity Catalog
# MAGIC
# MAGIC Este notebook **no modifica automáticamente** `OWNER TO`.
# MAGIC
# MAGIC Motivo:
# MAGIC
# MAGIC - `business_owner` representa accountability funcional;
# MAGIC - `data_steward` representa stewardship;
# MAGIC - `technical_owner` representa custodia técnica;
# MAGIC - `OWNER TO` requiere un principal/grupo real existente en el workspace.
# MAGIC
# MAGIC Ejemplo productivo:
# MAGIC
# MAGIC ```sql
# MAGIC ALTER TABLE workspace.yelp_gold.gld_review_risk
# MAGIC OWNER TO `grp_data_engineering`;
# MAGIC ```
# MAGIC
# MAGIC No se recomienda asignar `OWNER TO` directamente a una persona de negocio.

# COMMAND ----------

# MAGIC %md
# MAGIC # ✅ Definition of Done
# MAGIC
# MAGIC El notebook queda aprobado cuando:
# MAGIC
# MAGIC - [ ] todos los activos configurados existen;
# MAGIC - [ ] todos contienen `data_domain`;
# MAGIC - [ ] todos contienen `data_layer`;
# MAGIC - [ ] todos contienen `data_product`;
# MAGIC - [ ] todos contienen `business_owner`;
# MAGIC - [ ] todos contienen `data_steward`;
# MAGIC - [ ] todos contienen `technical_owner`;
# MAGIC - [ ] todos están trazados contra `raci_v1_4`;
# MAGIC - [ ] la validación contra `information_schema.table_tags` no presenta diferencias;
# MAGIC - [ ] el Quality Gate termina correctamente.
# MAGIC
# MAGIC ## Orden recomendado en el Job
# MAGIC
# MAGIC `06_gold_productos`
# MAGIC
# MAGIC ↓
# MAGIC `03_clasificacion_seguridad_uc`
# MAGIC
# MAGIC ↓
# MAGIC `04_ownership_stewardship_uc`
# MAGIC
# MAGIC ↓
# MAGIC `07_validacion_end_to_end`
